# DIRAC: VISUAL module overview

This is a second tutorial demonstrating the capabilities of the VISUAL module in DIRAC (for the introductory notebook, see `visual_interactive.ipynb`).

Here, we show how to integrate and/or visualize selected unperturbed and perturbed densities on grids for a few simple molecules. Note that we use precalculated densities, for which we used a poor setup for demonstration purposes. For production calculations, the computational model needs to be well calibrated.

The calculations presented in this notebook can also be invoked by running the `visual_tutorial_overview` test (`ctest -R visual_tutorial_overview`) in the `DIRAC` automatic test system.

Here, we use the `py3Dmol` library and ParaView for visualization and [RDKit](https://www.rdkit.org/) for molecular data manipulation, but any visualization software for plotting volumetric data can be used. We also demonstrate how to use different file formats, including HDF5, for which the `h5py` library is required. 
Please ensure you have all required libraries (see the "Prerequisites" section below). 

*Gosia Olejniczak, Warsaw University, 2025*

### Prerequisites: environment preparation

If you are using `conda` package manager, then you can create a working environment as follows:

```
cd conda_envs
conda env create -f dirac_visual.yml
cd ..
```

Make sure the environment is activated before running this notebook:

```
conda activate dirac_visual
```

If successful, you should be able to execute the next cell without errors:

In [ ]:
import os,sys
from pathlib import Path
import shutil
import subprocess
from IPython.display import display, HTML

import rdkit
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import Draw
from rdkit.Chem.Draw import IPythonConsole
Chem.Draw.IPythonConsole.ipython_3d = True
from rdkit import rdBase
from rdkit.Chem import rdmolfiles
from rdkit.Chem import rdDetermineBonds
from rdkit.Chem import rdMolAlign
from rdkit.Chem import rdMolTransforms

import py3Dmol

from ipywidgets import interact,fixed,IntSlider

Next, we set up the working directory for all files used or generated in this notebook:

In [ ]:
notebook_path=Path().parent.resolve()
dirac_path=notebook_path.parent.resolve()
dirac_testset=Path(dirac_path, 'test').resolve()

test_name='visual_tutorial_overview'
workdir=Path(Path.cwd(), 'workdir', test_name).resolve()
workdir.mkdir(parents=True, exist_ok=True)

#### Functions used in this tutorial

In [ ]:
def from_xyz_to_mol(fxyz):
    '''read xyz file, generate RDKit mol object'''
    raw_mol = rdmolfiles.MolFromXYZFile(str(fxyz))
    mol = Chem.Mol(raw_mol)
    rdDetermineBonds.DetermineConnectivity(mol)
    return mol

def view_single_mol(mol, add_volumetric=False, isovalue=0.1, cube_file=None):
    '''interactive visualization of mol (RDKit mol object)'''
    if add_volumetric:
        # mol + surface
        data = None
        with open(cube_file, "r") as infile:
            data = infile.read()
        view=py3Dmol.view(width=400, height=300)
        view.addVolumetricData(
            data,
            "cube",
            {
                "isoval": isovalue,
                "smoothness": 5,
                "opacity": 0.8,
                "volformat": "cube",
                "color": "blue",
            },
        )
        view.addModel(data, "cube")
        view.setStyle({"stick": {}, "sphere": {"scale": 0.3}})
        view.zoomTo()
    else:
        # only mol        
        view = py3Dmol.view(
            data=Chem.MolToMolBlock(mol),
            style={"stick": {}, "sphere": {"scale": 0.3}},
            width=300,
            height=300,
        )
    return view.show()

def dirac_output_file(dirname,finp,fmol):
    inp_string=str(finp).split('/')[-1].strip('.inp')
    mol_string=str(fmol).split('/')[-1].strip('.mol').strip('.xyz')       
    fout=Path(dirname, inp_string+'_'+mol_string+'.out').resolve()
    return str(fout)
    
def grep_from_dirac(pattern1, pattern2, file, exclude_last=False):
    if exclude_last:
        greplines="sed -n '/%s/,/^[^%s]/p' %s " %(pattern1, pattern2, file)
    else:
        greplines="sed -n '/%s/,/%s/p' %s " %(pattern1, pattern2, file)
    out=subprocess.run(greplines, shell=True, text=True, capture_output=True)
    print(out.stdout)

## Visualization of the unperturbed densities of the benzene molecule

In this section, we explore various unperturbed densities of the benzene molecule. We start with setting up the working environment: 

In [ ]:
os.chdir(str(workdir))
print('Demonstration directory: %s' %(workdir))

In [ ]:
dirac_test=Path(dirac_testset, test_name).resolve()
fmol=str(Path(dirac_test, 'c6h6.xyz').resolve())
finp1=str(Path(dirac_test, 'dc.inp').resolve())
finp2=str(Path(dirac_test, 'vis.inp').resolve())
results_dir=Path(dirac_test, 'result').resolve()

fout2=str(Path(dirac_test, 'result', 'vis_c6h6.out').resolve())

print('The input files are in the corresponding DIRAC test directory: %s' %(dirac_test))

Let us first look at the molecule:

In [ ]:
mol=from_xyz_to_mol(fmol)

In [ ]:
view_single_mol(mol)

The calculations were done in two steps in advance, but they can also be done as one job. First, we optimized the wave function and saved the result on the 'CHECKPOINT.h5' file, for which we use the `--outcmo` option:

In [ ]:
cmd = 'pam --inp=%s --mol=%s --noarch --nobackup --outcmo' % (finp1, fmol)
print('DIRAC run command (step 1): ')
print(cmd)

Then, we do calculations in the VISUAL module, using the generated 'CHECKPOINT.h5' file (the `--incmo` option).

### Visualization and integration of the electron density of water molecule

We use the capability of DIRAC to calculate many different densities on many different grids with the same input. In particular, we calculate the electron density (`name=ed`), and the electron localization function (`name=elf`).

The essential block in the visualization input is:

In [ ]:
grep_from_dirac('VISUAL', 'END', finp2)

Here, we specified two grids (`.GRIDS` section of the input): the 3D rectilinear grid with 64x64x32 points with a 4.0 a.u. margin added around the molecule (grid `id=1`) and the grid used in the DFT calculations in step 1 (grid `id=2`).
In the following examples, we will use the first grid for visualization purposes and the second one for integration.

The `.GRIDFUNCTION` section informs about the densities to be calculated on these grids. 

These calculations were run with the following command:

In [ ]:
cmd = 'pam --inp=%s --mol=%s --noarch --nobackup --incmo --get="*.cube"' % (finp2, fmol)
print('DIRAC run command (step 2): ')
print(cmd)

The first grid function specification line exports the electron density (`name=ed`) as a scalar field, which can be visualized (`purpose=visualization`). We export it in Gaussian `cube` format (`export=cube`) to the 'ed.cube' file (`file_out=ed.cube`). We visualize this density as a 3D surface at the isovalue=0.1:

In [ ]:
ed_cube=Path(results_dir, 'ed.cube').resolve()
view_single_mol(mol, add_volumetric=True, isovalue=0.1, cube_file=str(ed_cube))

We also integrate the electron density (`purpose=integration`), as declared in the second grid function specification in the input. To inspect the result of integration, we look in the DIRAC output file and search for the "Final GRIDFUNCTION values" printout:

In [ ]:
grep_from_dirac('Final grid function values (id=  2) integrated on grid (id=  2):', 'Elapsed', fout2, exclude_last=True)

As expected, the integrated value is close to 42, which is the number of electrons in benzene molecule.

### Visualization of the electron localization function (ELF) of water molecule

In the third line of the visualization input, we asked to export the ELF (`name=elf`) as a scalar field, on the same grid used for the electron density visualization, to the `elf.cube` file. We visualize this density as a 3D surface at the isovalue=0.8:

In [ ]:
elf_cube=Path(results_dir, 'elf.cube').resolve()
view_single_mol(mol, add_volumetric=True, isovalue=0.8, cube_file=str(elf_cube))

## Visualization of the perturbed densities of the benzene molecule: magnetic field perturbation

Here, we demonstrate how to visualize and integrate various densities related to magnetic field perturbations. Specifically, we focus on the magnetically induced current density (MICD) and the NMR shielding density in the benzene molecule, and further discuss them in more detail in the paragraphs below. 

We start with setting up the working environment:

In [ ]:
os.chdir(str(workdir))
print('Demonstration directory: %s' %(workdir))

In [ ]:
dirac_test=Path(dirac_testset, test_name).resolve()
fmol=str(Path(dirac_test, 'ch6h.xyz').resolve())
finp1=str(Path(dirac_test, 'dc.inp').resolve())
finp2=str(Path(dirac_test, 'rspB.inp').resolve())
finp3=str(Path(dirac_test, 'vis_rspB.inp').resolve())
results_dir=Path(dirac_test, 'result').resolve()

fout2=str(Path(dirac_test, 'result', 'rspB_c6h6.out').resolve())
fout3=str(Path(dirac_test, 'result', 'vis_rspB_c6h6.out').resolve())

print('The input files are in the corresponding DIRAC test directory: %s' %(dirac_test))


Both densities will be calculated with London atomic orbitals (LAOs) and simple magnetic balance in a few steps. 

First, we optimize the wave function ("finp1", step 1) and export the "CHECKPOINT.h5" file.
Then, we do the response calculations of the NMR shielding tensor ("finp2", step 2) and export the response data: "RESPONSE.h5" file with response vectors (this is the HDF5 equivalent of the "PAMXVC" file) and "TBMO.h5" file, which contains the details of the reorthonormalization procedure invoked when LAOs are used. This two-step procedure was presented in a separate tutorial (https://www.diracprogram.org/doc/release-25/tutorials/simple_magnetic_balance/tutorial.html)

In step 2, instead of calculating the NMR shielding, we could have specified the magnetic field perturbation explicitly with the `.OPERATOR` keyword (https://www.diracprogram.org/doc/release-25/manual/one_electron_operators.html#one-electron-operators). However, since the same perturbation operators are defined in the calculations of the NMR shielding tensor, we calculate them analytically in this step, and use them later for comparison with values from numerical integration ("finp3", step 3).

These DIRAC commands allow running the first two steps:

In [ ]:
cmd = 'pam --inp=%s --mol=%s --noarch --nobackup --outcmo' % (finp1, fmol)
print('DIRAC run command (step 1): ')
print(cmd)

In [ ]:
cmd = 'pam --inp=%s --mol=%s --noarch --nobackup --incmo --get="RESPONSE.h5 TBMO.h5"' % (finp2, fmol)
print('DIRAC run command (step 2): ')
print(cmd)

The visualization calculations can then be run as:

In [ ]:
cmd = 'pam --inp=%s --mol=%s --noarch --nobackup --incmo --put="RESPONSE.h5 TBMO.h5" --get="*.cube"' % (finp3, fmol)
print('DIRAC run command (step 3): ')
print(cmd)

Note that we calculate all perturbed densities discussed in this section in the same run:

In [ ]:
grep_from_dirac('VISUAL', 'END', finp3)

All grid functions are calculated with LAOs (`lao=1` keyword). 
Also, in all cases, we are interested in the response to the "z"-component of the magnetic field. First, we need to identify the index of the corresponding response vectors on the "RESPONSE.h5" file. Using the `h5dump RESPONSE.h5` to view the content of this file, we search for the "lab" dataset whose value equals "LAO-ZRM1H1". This is the case for data groups "vec_5" and "vec_6". The response vectors in the group "vec_5" have the "typ" dataset with the value "E-", while the  vectors in the group "vec_6" have the "typ" dataset with the value "P-".
Vectors in "vec_5" involve the "e-e" orbital rotations and are responsible for the paramagnetic contribution to magnetic response, while vectors in "vec_6" involve the "e-p" rotations that contribute to the diamagnetic part of MICD and NMR shielding tensors.

Since we want to include all contributions to the property densities, we define two grid functions in the input for each density we are interested in, one with the `rspvec_id=5` and the other with the `rspvec_id=6` keywords; these point to response vectors of indices 5 and 6 on the "RESPONSE.h5" file (`rspvec_file=RESPONSE.h5`). The appropriate data on `TBMO.h5` file is then used.

In the following paragraphs, we further discuss the visualization of the magnetically induced current density and the integration of the NMR shielding density.

### Visualization of the magnetically-induced current density


Here, we demonstrate how to visualize the density induced by the "z"-component of the magnetic field ($B_z$). 
Due to the use of LAOs, these densities are calculated according to the following formula:

$$
\vec{j}^{B_z} = 
       \underbrace{-\frac{i}{2}e(\vec{R}_{MN} \times \vec{r})_z D_{\nu\mu} \Omega_{\mu\nu}^{c\alpha}}_{\vec{j}^{B_z}_{direct}}
       \quad
       \underbrace{-e(c_{\mu p^*} T_{pi}^{B_z*} c_{\nu i} + c_{\nu p} T_{pi}^{B_z} c_{\mu i}^*)  \Omega_{\mu\nu}^{c\alpha}}_{\vec{j}^{B_z}_{reorth}}
       \quad       
      \underbrace{-e(c_{\mu s^*} \kappa_{si}^{B_z*} c_{\nu i} + c_{\nu s} \kappa_{si}^{B_z} c_{\mu i}^*)  \Omega_{\mu\nu}^{c\alpha}}_{\vec{j}^{B_z}_{response}}
$$


Here, $D_{\nu\mu}$ is the unperturbed density matrix, $\Omega_{\mu\nu}^{c\alpha} = \chi_\mu ^\dagger c\vec{\alpha} \chi_\nu$ is the velocity distribution of atomic orbitals $\chi_\mu$  and $\chi_\nu$, $T_{pi}^{B_z}$ are connection matrices and $\kappa_{si}^{B_z}$ are the orbital rotation amplitudes. The details are in the publication (DOI: 10.1039/C1CP22457C).


The relevant lines in the `.GRIDFUNCTION` specification in the user input contain the keyword `name=j`. The use of LAOs in the visualization calculations means that these densities already include all three contributions: from the response ($\vec{j}^{B_z}_{response}$) and reorthornomalization ($\vec{j}^{B_z}_{reorth}$) terms, and London phase factors ($\vec{j}^{B_z}_{direct}$). The expert keyword `lao_skip` allows the exclusion of the selected contributions.

In particular, the first grid function in the input:

`name=j id_grid=1 purpose=visualization rspvec_file=RESPONSE.h5 rspvec_id=5 lao=1 export=csv file_out=j_Bz_lao_rspvec5.csv`

calculates the 'e-e' contributions (as explained above) to the response and reorthonormalization part; the "direct" contribution is also included.

The second grid function in the input:

`name=j id_grid=1 purpose=visualization rspvec_file=RESPONSE.h5 rspvec_id=6 lao=1 export=csv file_out=j_Bz_lao_rspvec6.csv`

calculates the 'e-p' contributions (as explained above) to the response and reorthonormalization part; the "direct" contribution is also included.


To visualize the **total** magnetically induced current density vector (induced by $B_z$), we need to sum vectors exported on files `j_Bz_lao_rspvec5.csv` and `j_Bz_lao_rspvec6.csv`. Below, we draw the vector field corresponding to this sum; it is plotted in the plane perpendicular to the "z" direction, and shifted by 0.5 a.u. above the molecular plane (plot prepared in ParaView):

![test](visual_data/static/micd_c6h6.png)

### The NMR shielding density


Here, we focus on the NMR shielding density of the carbon nucleus in the benzene molecule. 
The NMR shielding density can be formulated in terms of the magnetically induced current density (the integrand of the following expression) and integrated to the values of the elements of the NMR shielding tensor:

$$
\sigma_{\alpha\beta}^K = - \frac{1}{c^2} \int \frac{1}{\vec{r}_K^3} \left( \vec{r}_K \times \vec{j}^{B_\alpha} \right)_\beta d\vec{r}
$$

where $K$ denotes the nucleus of interest at $\vec{R}_K$, and $\vec{r}_K = |\vec{r} - \vec{R}_K|$.
The formula for the magnetically induced current density, $\vec{j}^{B_\alpha}$, in this case calculated with LAOs for $\alpha=z$ was demonstrated in the previous paragraph.

The calculation of the NMR shielding density is activated with the `name=ndip` keyword in the input file (the third and fourth lines in the `.GRIDFUNCTION` specification in "finp3", invoking the calculations of the "e-e" and "e-p" contributions to this density, as explained above):

```
name=ndip id_grid=2 purpose=integration rspvec_file=RESPONSE.h5 rspvec_id=5 lao=1 nuc=1 uscale=-53.25135
name=ndip id_grid=2 purpose=integration rspvec_file=RESPONSE.h5 rspvec_id=6 lao=1 nuc=1 uscale=-53.25135
```

For a fixed perturbation operator ($\alpha=z$), the NMR shielding density is a vector whose components ($\beta$) correspond to the nuclear dipole moment operator in "x", "y", and "z" directions.

Here, we are interested in nucleus 1 (ordering as in the xyz file); the index of a nucleus is provided with the `nuc=1` keyword. 
To integrate this density on the DFT numerical grid (`id_grid=2`) and obtain the result in ppm, we need to scale it with -53.25135 (`uscale=-53.25135`). 

The result of these calculations is printed in the output ("vis_rspB_c6h6.out") in the following lines:

```
  Final grid function values (id=  3) integrated on grid (id=  2):
  -4.29847810E-007   8.37778701E-007   4.31463484E+001
```

and

```
  Final grid function values (id=  4) integrated on grid (id=  2):
  -5.83157504E-008   3.97631832E-008   1.33433229E+002
```

We can now compare the integrated values of the NMR shielding with the analytical values calculated in step 2, for the first carbon nucleus. 

For example, the **total** numerical value of the "zz" component of the NMR shielding tensor is **43.1463 ppm + 133.43323 ppm = 176.5796 ppm**.

Note that the use of LAOs in the visualization calculations means that the integrated value already has contributions from the reorthornomalization terms and London phase factors; therefore, we can compare it only with the **total** value of the "zz" component of the NMR shielding calculated analytically in step 2, which was **176.5796 ppm**, as can be identified from the "rspB_c6h6.out" output (pasted below). The accuracy of the analytical and numerical values up to 4 digits is excellent. 

```
  Chemical shielding for C     :
  ==============================


  Shielding constant:     55.1150 ppm
  Anisotropy:            182.1969 ppm
  Span:                  231.5834 ppm
  Asymmetry:               0.8132
  Skew:                    0.1470


  Total shielding tensor (ppm):
  -----------------------------

                  Bx             By             Bz

  C    x    43.76918740    -0.00000041    -0.00000036
  C    y     0.00000013   -55.00378790     0.00000095
  C    z     0.00000007    -0.00000039   176.57958741

```

This density can also be visualized. For this, the user input needs to be adapted (calculations not shown here).

### What else is there

See the `DIRAC` tutorial for more VISUAL options

### Good practices and gotchas

Remember that the molecule can get reoriented if symmetry is detected
automatically.

When plotting perturbed densities it is important to use CHECKPOINT.h5 and RESPONSE.h5 from
the same run. During the visualization step the wave function should not be
allowed to (re)optimize.

When plotting orbital densities it is useful to do a 3D integration
at the same time. As a sanity check the integrated number should be twice the
number of orbitals.